# ADM NB Explained

This notebook shows exactly how all the values in an ADM model report
are calculated. It also shows how the propensity is calculated for a
particular customer.

We use one of the shipped datamart exports for the example. This is a
model very similar to one used in some of the ADM PowerPoint/Excel deep
dive examples. You can change this notebook to apply to your own data.


> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm`, `healthcheck` extras. Install with your favorite package manager, e.g. `pip install "pdstools[adm,healthcheck]"`.


In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately

import plotly.io as pio

pio.renderers.default = "notebook_connected"


In [2]:
import polars as pl
import numpy as np

import plotly.express as px
from math import log
from great_tables import GT
from pdstools import datasets
from pdstools.utils import cdh_utils

In [3]:
model_name = "AutoNew84Months"
predictor_name = "Customer.NetWealth"
channel = "Web"

For the example we pick one particular model over a channel.
To explain the ADM model report, we use one of the active predictors as an
example. Swap for any other predictor when using different data.

In [4]:
dm = datasets.cdh_sample()

model = dm.combined_data.filter(
    (pl.col("Name") == model_name) & (pl.col("Channel") == channel)
)

modelpredictors = (
    dm.combined_data.join(
        model.select(pl.col("ModelID").unique()), on="ModelID", how="inner"
    )
    .filter(pl.col("EntryType") != "Inactive")
    .with_columns(
        Action=pl.concat_str(["Issue", "Group"], separator="/"),
        PredictorName=pl.col("PredictorName").cast(pl.Utf8),
    )
    .collect()
)

predictorbinning = modelpredictors.filter(
    pl.col("PredictorName") == predictor_name
).sort("BinIndex")

In [5]:
model_id = None

if (modelpredictors.select(pl.col("ModelID").unique()).shape[0] > 1) and (
    model_id is None
):
    display(
        model.group_by("ModelID")
        .agg(
            number_of_predictors=pl.col("PredictorName").n_unique(),
            model_performance=cdh_utils.weighted_performance_polars() * 100,
            response_count=pl.sum("ResponseCount"),
        )
        .collect()
    )
    raise Exception(
        f"**{model_name}** model has multiple instances."
        "\nThis could be due to the same model name being used in different configurations, directions, issues, or having multiple treatments."
        "\nTo ensure the selection of a unique model, please choose a model_id from the table above and update the `model_id` variable at the top of this cell."
        "\nAfterward, rerun this cell."
        f"\nSee model IDs in {model_name} model above:"
    )
if model_id is not None:
    if (
        model_id
        not in modelpredictors.select(pl.col("ModelID").unique())
        .get_column("ModelID")
        .to_list()
    ):
        raise Exception(
            f"The {model_name} model does not have a model ID: {model_id}."
            f"Please ensure that the spelling of the model ID is correct."
            f"You can run `modelpredictors.select(pl.col('ModelID').unique().implode()).row(0)` to see the exact spellings of your IDs."
            "After updating the `model_id`, you can restart the notebook from the beginning."
        )

    predictors_in_selected_model = (
        modelpredictors.filter(pl.col("ModelID") == model_id)
        .select(pl.col("PredictorName").unique())
        .get_column("PredictorName")
        .to_list()
    )
    if predictor_name not in predictors_in_selected_model:
        raise Exception(
            f"{predictor_name} is not a predictor of the model with ID: {model_id}."
            "Please choose one of the available predictors below and update the **predictor_name** variable in the cell above:"
            f"\nAvailable Predictors:\n{predictors_in_selected_model}."
        )

    modelpredictors = modelpredictors.filter(pl.col("ModelID") == model_id)
    predictorbinning = predictorbinning.filter(pl.col("ModelID") == model_id)
    print(f"{model_name} model with **{model_id}** model ID is selected successfully.")

## Model Overview

The selected model is shown below. Only the currently active predictors are used for the propensity calculation, so only showing those.



In [6]:
from great_tables import loc, style

GT(
    modelpredictors.select(
        pl.col("Action").unique(),
        pl.col("Channel").unique(),
        pl.col("Name").unique(),
        pl.col("PredictorName")
        .unique()
        .sort()
        .implode()
        .list.join(", ")
        .alias("Active Predictors"),
        (pl.col("Performance").unique() * 100).alias("Model Performance (AUC)"),
    ).transpose(include_header=True)
).tab_header("Overview").tab_options(column_labels_hidden=True).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="column")
)

GT(_tbl_data=shape: (5, 2)
┌─────────────────────────┬─────────────────────────────────┐
│ column                  ┆ column_0                        │
│ ---                     ┆ ---                             │
│ str                     ┆ str                             │
╞═════════════════════════╪═════════════════════════════════╡
│ Action                  ┆ Sales/AutoLoans                 │
│ Channel                 ┆ Web                             │
│ Name                    ┆ AutoNew84Months                 │
│ Active Predictors       ┆ Classifier, Customer.Age, Cust… │
│ Model Performance (AUC) ┆ 77.4901                         │
└─────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x7f1e2c00e790>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='column_0', type=<ColInfoTypeEnum.default: 1>, column_label='column_0', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1def224f50>, _spanners=Spanners([]), _heading=Heading(title='Overview', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f1def224f90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f1dee8ae150>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=2, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=3, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=4, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x7f1e2c7f6290>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', typ

## Binning of the selected Predictor

The Model Report in Prediction Studio for this model will have a predictor binning plot like below.

All numbers can be derived from just the number of positives and negatives in each bin that are stored in the ADM Data Mart. The next sections will show exactly how that is done.

In [7]:
display(
    GT(
        (
            predictorbinning.group_by("PredictorName")
            .agg(
                pl.first("ResponseCount").cast(pl.Utf8).alias("# Responses"),
                pl.n_unique("BinIndex").cast(pl.Utf8).alias("# Bins"),
                (pl.first("PredictorPerformance") * 100)
                .cast(pl.Utf8)
                .alias("Predictor Performance(AUC)"),
            )
            .rename({"PredictorName": "Predictor Name"})
            .transpose(include_header=True)
        )
    )
    .tab_header("Predictor information")
    .tab_options(column_labels_hidden=True)
    .tab_style(style=style.text(weight="bold"), locations=loc.body(columns="column"))
    .tab_options(table_margin_left=0)
)

fig = dm.plot.predictor_binning(
    model_id=modelpredictors.get_column("ModelID").unique().to_list()[0],
    predictor_name=predictor_name,
)
fig.update_layout(width=600, height=400)


GT(_tbl_data=shape: (4, 2)
┌────────────────────────────┬────────────────────┐
│ column                     ┆ column_0           │
│ ---                        ┆ ---                │
│ str                        ┆ str                │
╞════════════════════════════╪════════════════════╡
│ Predictor Name             ┆ Customer.NetWealth │
│ # Responses                ┆ 1636.0             │
│ # Bins                     ┆ 8                  │
│ Predictor Performance(AUC) ┆ 72.2077            │
└────────────────────────────┴────────────────────┘, _body=<great_tables._gt_data.Body object at 0x7f1def0a2e90>, _boxhead=Boxhead([ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None), ColInfo(var='column_0', type=<ColInfoTypeEnum.default: 1>, column_label='column_0', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1def0a2a50>, _spanners=Spanners([]), _heading=Heading(title='Predictor information', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f1def0a3190>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f1def0a3090>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=2, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='column', rows=None, mask=None), grpname=None, colname='column', rownum=3, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x7f1def0a3010>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value=0), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='tab

In [8]:
BinPositives = pl.col("BinPositives")
BinNegatives = pl.col("BinNegatives")
sumPositives = pl.sum("BinPositives")
sumNegatives = pl.sum("BinNegatives")

binstats = predictorbinning.select(
    pl.col("BinSymbol").alias("Range/Symbol"),
    ((BinPositives + BinNegatives) / (sumPositives + sumNegatives)).alias(
        "Responses (%)"
    ),
    BinPositives.cast(pl.Int32).alias("Positives"),
    (BinPositives / sumPositives).alias("Positives (%)"),
    BinNegatives.cast(pl.Int32).alias("Negatives"),
    (BinNegatives / sumNegatives).alias("Negatives (%)"),
    (BinPositives / (BinPositives + BinNegatives)).alias("Propensity (%)"),
    cdh_utils.z_ratio(neg_col=BinNegatives, pos_col=BinPositives),
    (
        (BinPositives / (BinPositives + BinNegatives))
        / (sumPositives / (BinPositives + BinNegatives).sum())
    ).alias("Lift"),
)
total_positives = binstats.select(pl.sum("Positives")).item()
total_negatives = binstats.select(pl.sum("Negatives")).item()
total_row = {
    "Range/Symbol": "Total",
    "Responses (%)": binstats.select(pl.sum("Responses (%)")).item(),
    "Positives": total_positives,
    "Positives (%)": binstats.select(pl.sum("Positives (%)")).item(),
    "Negatives": total_negatives,
    "Negatives (%)": binstats.select(pl.sum("Negatives (%)")).item(),
    "Propensity (%)": total_positives / (total_positives + total_negatives),
    "ZRatio": 0.0,
    "Lift": 1.0,
}

total_df = pl.DataFrame(total_row, schema=binstats.schema)
binstats = binstats.vstack(total_df)


GT(binstats).tab_header("Binning statistics").tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).fmt_percent(pl.selectors.ends_with("(%)")).fmt_number(["ZRatio", "Lift"]).tab_options(
    table_margin_left=0
).tab_options(table_margin_left=0)



GT(_tbl_data=shape: (9, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Range/Sym ┆ Responses ┆ Positives ┆ Positives ┆ … ┆ Negatives ┆ Propensit ┆ ZRatio    ┆ Lift     │
│ bol       ┆ (%)       ┆ ---       ┆ (%)       ┆   ┆ (%)       ┆ y (%)     ┆ ---       ┆ ---      │
│ ---       ┆ ---       ┆ i32       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f32       ┆ f32      │
│ str       ┆ f32       ┆           ┆ f32       ┆   ┆ f32       ┆ f32       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ <11684.56 ┆ 0.266504  ┆ 13        ┆ 0.063107  ┆ … ┆ 0.295804  ┆ 0.029817  ┆ -11.18687 ┆ 0.236795 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ 7         ┆          │
│ [11684.56 ┆ 0.123472  ┆ 24        ┆ 0.116505  ┆ … ┆ 0.124476  ┆ 0.118812  ┆ -0.332147 ┆ 0.943574 │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 13732.56> ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ [13732.56 ┆ 0.163203  ┆ 17        ┆ 0.082524  ┆ … ┆ 0.174825  ┆ 0.06367   ┆ -4.264671 ┆ 0.505654 │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 16845.52> ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ [16845.52 ┆ 0.140587  ┆ 51        ┆ 0.247573  ┆ … ┆ 0.125175  ┆ 0.221739  ┆ 3.908162  ┆ 1.760996 │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 19139.28> ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ [19139.28 ┆ 0.055012  ┆ 7         ┆ 0.033981  ┆ … ┆ 0.058042  ┆ 0.077778  ┆ -1.711776 ┆ 0.617692 │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 20286.16> ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ [20286.16 ┆ 0.135697  ┆ 53        ┆ 0.257282  ┆ … ┆ 0.118182  ┆ 0.238739  ┆ 4.397646  ┆ 1.896003 │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 22743.76> ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ [22743.76 ┆ 0.055012  ┆ 13        ┆ 0.063107  ┆ … ┆ 0.053846  ┆ 0.144444  ┆ 0.515565  ┆ 1.147141 │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 23890.64> ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ >=23890.6 ┆ 0.060513  ┆ 28        ┆ 0.135922  ┆ … ┆ 0.04965   ┆ 0.282828  ┆ 3.512888  ┆ 2.246151 │
│ 4         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ Total     ┆ 1.0       ┆ 206       ┆ 1.0       ┆ … ┆ 1.0       ┆ 0.125917  ┆ 0.0       ┆ 1.0      │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x7f1dee2faad0>, _boxhead=Boxhead([ColInfo(var='Range/Symbol', type=<ColInfoTypeEnum.default: 1>, column_label='Range/Symbol', column_align='left', column_width=None), ColInfo(var='Responses (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Responses (%)', column_align='right', column_width=None), ColInfo(var='Positives', type=<ColInfoTypeEnum.default: 1>, column_label='Positives', column_align='right', column_width=None), ColInfo(var='Positives (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Positives (%)', column_align='right', column_width=None), ColInfo(var='Negatives', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives', column_align='right', column_width=None), ColInfo(var='Negatives (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives (%)', column_align='right', column_width=None), ColInfo(var='Propensity (%)', type=<ColInfoTypeEnum.default: 1>, c

## Bin Statistics

### Positive and Negative ratios

Internally, ADM only keeps track of the total counts of positive and negative responses in each bin. Everything else is derived from those numbers. The percentages and totals are trivially derived, and the propensity is just the number of positives divided by the total. The numbers calculated here match the numbers from the datamart table exactly.

In [9]:
binning_derived = predictorbinning.select(
    pl.col("BinSymbol").alias("Range/Symbol"),
    BinPositives.alias("Positives"),
    BinNegatives.alias("Negatives"),
    ((BinPositives + BinNegatives) / (sumPositives + sumNegatives)).alias(
        "Responses %"
    ),
    (BinPositives / sumPositives).alias("Positives %"),
    (BinNegatives / sumNegatives).alias("Negatives %"),
    (BinPositives / (BinPositives + BinNegatives)).round(4).alias("Propensity"),
)

pcts = ["Responses %", "Positives %", "Negatives %", "Propensity"]
GT(binning_derived).tab_header("Derived binning statistics").tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"),
    locations=loc.body(columns=pcts),
).fmt_percent(pcts).tab_options(table_margin_left=0)


GT(_tbl_data=shape: (8, 7)
┌──────────────┬───────────┬───────────┬─────────────┬─────────────┬─────────────┬────────────┐
│ Range/Symbol ┆ Positives ┆ Negatives ┆ Responses % ┆ Positives % ┆ Negatives % ┆ Propensity │
│ ---          ┆ ---       ┆ ---       ┆ ---         ┆ ---         ┆ ---         ┆ ---        │
│ str          ┆ f32       ┆ f32       ┆ f32         ┆ f32         ┆ f32         ┆ f32        │
╞══════════════╪═══════════╪═══════════╪═════════════╪═════════════╪═════════════╪════════════╡
│ <11684.56    ┆ 13.0      ┆ 423.0     ┆ 0.266504    ┆ 0.063107    ┆ 0.295804    ┆ 0.0298     │
│ [11684.56,   ┆ 24.0      ┆ 178.0     ┆ 0.123472    ┆ 0.116505    ┆ 0.124476    ┆ 0.1188     │
│ 13732.56>    ┆           ┆           ┆             ┆             ┆             ┆            │
│ [13732.56,   ┆ 17.0      ┆ 250.0     ┆ 0.163203    ┆ 0.082524    ┆ 0.174825    ┆ 0.0637     │
│ 16845.52>    ┆           ┆           ┆             ┆             ┆             ┆            │
│ [16845.52,   ┆ 51.0      ┆ 179.0     ┆ 0.140587    ┆ 0.247573    ┆ 0.125175    ┆ 0.2217     │
│ 19139.28>    ┆           ┆           ┆             ┆             ┆             ┆            │
│ [19139.28,   ┆ 7.0       ┆ 83.0      ┆ 0.055012    ┆ 0.033981    ┆ 0.058042    ┆ 0.0778     │
│ 20286.16>    ┆           ┆           ┆             ┆             ┆             ┆            │
│ [20286.16,   ┆ 53.0      ┆ 169.0     ┆ 0.135697    ┆ 0.257282    ┆ 0.118182    ┆ 0.2387     │
│ 22743.76>    ┆           ┆           ┆             ┆             ┆             ┆            │
│ [22743.76,   ┆ 13.0      ┆ 77.0      ┆ 0.055012    ┆ 0.063107    ┆ 0.053846    ┆ 0.1444     │
│ 23890.64>    ┆           ┆           ┆             ┆             ┆             ┆            │
│ >=23890.64   ┆ 28.0      ┆ 71.0      ┆ 0.060513    ┆ 0.135922    ┆ 0.04965     ┆ 0.2828     │
└──────────────┴───────────┴───────────┴─────────────┴─────────────┴─────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x7f1def084110>, _boxhead=Boxhead([ColInfo(var='Range/Symbol', type=<ColInfoTypeEnum.default: 1>, column_label='Range/Symbol', column_align='left', column_width=None), ColInfo(var='Positives', type=<ColInfoTypeEnum.default: 1>, column_label='Positives', column_align='right', column_width=None), ColInfo(var='Negatives', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives', column_align='right', column_width=None), ColInfo(var='Responses %', type=<ColInfoTypeEnum.default: 1>, column_label='Responses %', column_align='right', column_width=None), ColInfo(var='Positives %', type=<ColInfoTypeEnum.default: 1>, column_label='Positives %', column_align='right', column_width=None), ColInfo(var='Negatives %', type=<ColInfoTypeEnum.default: 1>, column_label='Negatives %', column_align='right', column_width=None), ColInfo(var='Propensity', type=<ColInfoTypeEnum.default: 1>, column_label='Propensity', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1dee317c90>, _spanners=Spanners([]), _heading=Heading(title='Derived binning statistics', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f1def059b10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f1dee31d5d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows=None, mask=None), grpname=None, colname='Range/Symbol', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Range/Symbol', rows

### Lift

Lift is the ratio of the propensity in a particular bin over the average propensity. So a value of 1 is the average, larger than 1 means higher propensity, smaller means lower propensity:

In [10]:
positives = pl.col("Positives")
negatives = pl.col("Negatives")
sumPositives = pl.sum("Positives")
sumNegatives = pl.sum("Negatives")
GT(
    binning_derived.select(
        "Range/Symbol",
        "Positives",
        "Negatives",
        (
            (positives / (positives + negatives))
            / (sumPositives / (positives + negatives).sum())
        )
        .round(4)
        .alias("Lift"),
    )
).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"), locations=loc.body(columns=["Lift"])
).tab_options(table_margin_left=0)

Range/Symbol,Positives,Negatives,Lift
<11684.56,13.0,423.0,0.2368
"[11684.56, 13732.56>",24.0,178.0,0.9436
"[13732.56, 16845.52>",17.0,250.0,0.5057
"[16845.52, 19139.28>",51.0,179.0,1.761
"[19139.28, 20286.16>",7.0,83.0,0.6177
"[20286.16, 22743.76>",53.0,169.0,1.896
"[22743.76, 23890.64>",13.0,77.0,1.1471
>=23890.64,28.0,71.0,2.2462


### Z-Ratio

The Z-Ratio is also a measure of the how the propensity in a bin differs from the average, but takes into account the size of the bin and thus is statistically more relevant. It represents the number of standard deviations from the average, so centers around 0. The wider the spread, the better the predictor is.
$$\frac{posFraction-negFraction}{\sqrt(\frac{posFraction*(1-posFraction)}{\sum positives}+\frac{negFraction*(1-negFraction)}{\sum negatives})}$$ 

See the calculation here, which is also included in [cdh_utils' z_ratio()](https://pegasystems.github.io/pega-datascientist-tools/autoapi/pdstools/utils/cdh_utils/index.html#pdstools.utils.cdh_utils.z_ratio).

In [11]:
def z_ratio(
    pos_col: pl.Expr = pl.col("BinPositives"), neg_col: pl.Expr = pl.col("BinNegatives")
) -> pl.Expr:
    def get_fracs(pos_col=pl.col("BinPositives"), neg_col=pl.col("BinNegatives")):
        return pos_col / pos_col.sum(), neg_col / neg_col.sum()

    def z_ratio_impl(
        pos_fraction_col=pl.col("posFraction"),
        neg_fraction_col=pl.col("negFraction"),
        positives_col=pl.sum("BinPositives"),
        negatives_col=pl.sum("BinNegatives"),
    ):
        return (
            (pos_fraction_col - neg_fraction_col)
            / (
                (pos_fraction_col * (1 - pos_fraction_col) / positives_col)
                + (neg_fraction_col * (1 - neg_fraction_col) / negatives_col)
            ).sqrt()
        ).alias("ZRatio")

    return z_ratio_impl(*get_fracs(pos_col, neg_col), pos_col.sum(), neg_col.sum())


GT(
    binning_derived.select(
        "Range/Symbol", "Positives", "Negatives", "Positives %", "Negatives %"
    ).with_columns(z_ratio(positives, negatives).round(4))
).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"), locations=loc.body(columns=["ZRatio"])
).fmt_percent(pl.selectors.ends_with("%")).tab_options(table_margin_left=0)

Range/Symbol,Positives,Negatives,Positives %,Negatives %,ZRatio
<11684.56,13.0,423.0,6.31%,29.58%,-11.1869
"[11684.56, 13732.56>",24.0,178.0,11.65%,12.45%,-0.3321
"[13732.56, 16845.52>",17.0,250.0,8.25%,17.48%,-4.2647
"[16845.52, 19139.28>",51.0,179.0,24.76%,12.52%,3.9082
"[19139.28, 20286.16>",7.0,83.0,3.40%,5.80%,-1.7118
"[20286.16, 22743.76>",53.0,169.0,25.73%,11.82%,4.3976
"[22743.76, 23890.64>",13.0,77.0,6.31%,5.38%,0.5156
>=23890.64,28.0,71.0,13.59%,4.97%,3.5129


## Predictor AUC


The predictor AUC is the univariate performance of this predictor against the outcome. This too can be derived from the positives and negatives and
there is  a convenient function in pdstools to calculate it directly from the positives and negatives.

This function is implemented in cdh_utils: [cdh_utils.auc_from_bincounts()](https://pegasystems.github.io/pega-datascientist-tools/autoapi/pdstools/utils/cdh_utils/index.html#pdstools.utils.cdh_utils.auc_from_bincounts).

In [12]:
pos = binning_derived.get_column("Positives")
neg = binning_derived.get_column("Negatives")
probs = binning_derived.get_column("Propensity")
order = probs.arg_sort()
FPR = pl.Series([0.0], dtype=pl.Float32).extend(neg[order].cum_sum() / neg[order].sum())
TPR = pl.Series([0.0], dtype=pl.Float32).extend(pos[order].cum_sum() / pos[order].sum())
if TPR[1] < 1 - FPR[1]:
    FPR, TPR = TPR, FPR


In [13]:
pos = binning_derived.get_column("Positives").to_numpy()
neg = binning_derived.get_column("Negatives").to_numpy()
probs = binning_derived.get_column("Propensity").to_numpy()
order = np.argsort(probs)

FPR = np.cumsum(neg[order]) / np.sum(neg[order])
TPR = np.cumsum(pos[order]) / np.sum(pos[order])
TPR = np.insert(TPR, 0, 0, axis=0)
FPR = np.insert(FPR, 0, 0, axis=0)
# Checking whether classifier labels are correct
if TPR[1] < 1 - FPR[1]:
    temp = FPR
    FPR = TPR
    TPR = temp
auc = cdh_utils.auc_from_bincounts(pos=pos, neg=neg, probs=probs)

fig = px.line(
    x=[1 - x for x in FPR],
    y=TPR,
    labels=dict(x="Specificity", y="Sensitivity"),
    title=f"AUC = {auc.round(3)}",
    width=700,
    height=700,
    range_x=[1, 0],
    template="none",
)
fig.add_shape(type="line", line=dict(dash="dash"), x0=1, x1=0, y0=0, y1=1)
fig.show()


## Predictor Contributions

### Naive Bayes and Log Odds

The basis for the Naive Bayes algorithm is Bayes' Theorem:

$$p(C_k|x) = \frac{p(x|C_k)*p(C_k)}{p(x)}$$

with $C_k$ the outcome and $x$ the customer. Bayes' theorem turns the
question "what's the probability to accept this action given a customer" around to 
"what's the probability of this customer given an action". With the independence
assumption, and after applying a log odds transformation we get a log odds score 
that can be calculated efficiently and in a numerically stable manner:

$$log\ odds\ score = \sum_{p\ \in\ active\ predictors}log(p(x_p|Positive)) + log(p_{positive}) - \sum_plog(p(x_p|Negative)) - log(p_{negative})$$
note that the _prior_ can be written as:

$$log(p_{positive}) - log(p_{negative}) = log(\frac{TotalPositives}{Total})-log(\frac{TotalNegatives}{Total}) = log(TotalPositives) - log(TotalNegatives)$$


### Predictor Contribution

The contribution (_conditional log odds_) of an active predictor $p$ for bin $i$ with the number
of positive and negative responses in $Positives_i$ and $Negatives_i$ is calculated as (note the "Laplace smoothing" to avoid log 0 issues):

$$contribution_p = \log(Positives_i+\frac{1}{nBins}) - \log(Negatives_i+\frac{1}{nBins}) - \log(1+\sum_{i\ = 1..nBins}{Positives_i}) + \log(1+\sum_i{Negatives_i})$$


In [14]:
N = binning_derived.shape[0]
GT(
    binning_derived.with_columns(
        LogOdds=(pl.col("Positives %") / pl.col("Negatives %")).log().round(5),
        ModifiedLogOdds=(
            ((positives + 1 / N).log() - (positives.sum() + 1).log())
            - ((negatives + 1 / N).log() - (negatives.sum() + 1).log())
        ).round(5),
    ).drop("Responses %", "Propensity")
).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Range/Symbol")
).tab_style(
    style=style.text(color="blue"),
    locations=loc.body(columns=["LogOdds", "ModifiedLogOdds"]),
).fmt_percent(pl.selectors.ends_with("%")).tab_options(table_margin_left=0)

Range/Symbol,Positives,Negatives,Positives %,Negatives %,LogOdds,ModifiedLogOdds
<11684.56,13.0,423.0,6.31%,29.58%,-1.54487,-1.53974
"[11684.56, 13732.56>",24.0,178.0,11.65%,12.45%,-0.06618,-0.06583
"[13732.56, 16845.52>",17.0,250.0,8.25%,17.48%,-0.75069,-0.74801
"[16845.52, 19139.28>",51.0,179.0,24.76%,12.52%,0.68199,0.6796
"[19139.28, 20286.16>",7.0,83.0,3.40%,5.80%,-0.53538,-0.52333
"[20286.16, 22743.76>",53.0,169.0,25.73%,11.82%,0.77795,0.77542
"[22743.76, 23890.64>",13.0,77.0,6.31%,5.38%,0.1587,0.1625
>=23890.64,28.0,71.0,13.59%,4.97%,1.00708,1.00563


## Propensity mapping

### Log odds contribution for all the predictors

The final score is loosely referred to as "the average contribution" but
in fact is a little more nuanced. The final score is calculated as:

$$score = \frac{\log(1 + TotalPositives) – \log(1 + TotalNegatives) + \sum_p contribution_p}{1 + nActivePredictors}$$

Here, $TotalPositives$ and $TotalNegatives$ are the total number of
positive and negative responses to the model.

Below an example. From all the active predictors of the model 
we pick a value (in the middle for numerics, first symbol
for symbolics) and show the (modified) log odds.


In [15]:
def middle_bin():
    return pl.col("BinIndex") == (pl.max("BinIndex") / 2).floor().cast(pl.UInt32)


if not all(
    col in modelpredictors.columns for col in ["BinLowerBound", "BinUpperBound"]
):

    def extract_numbers_in_contents(s: str, index):
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", s)
        try:
            number = float(numbers[index])
        except:
            number = 0
        return number

    modelpredictors = modelpredictors.with_columns(
        pl.col("Contents").cast(pl.Utf8)
    ).with_columns(
        pl.when(pl.col("Type") == "numeric")
        .then(
            pl.col("Contents").map_elements(
                lambda col: extract_numbers_in_contents(col, 0)
            )
        )
        .otherwise(pl.lit(-9999))
        .alias("BinLowerBound")
        .cast(pl.Float32),
        pl.when(pl.col("Type") == "numeric")
        .then(
            pl.col("Contents").map_elements(
                lambda col: extract_numbers_in_contents(col, 1)
            )
        )
        .otherwise(pl.lit(-9999))
        .alias("BinUpperBound")
        .cast(pl.Float32),
    )


def row_wise_log_odds(bin, positives, negatives):
    bin, N = bin.list.get(0) - 1, positives.list.len()
    pos, neg = positives.list.get(bin), negatives.list.get(bin)
    pos_sum, neg_sum = positives.list.sum(), negatives.list.sum()
    return (
        ((pos + (1 / N)).log() - (pos_sum + 1).log())
        - (((neg + (1 / N)).log()) - (neg_sum + 1).log())
    ).alias("Modified Log odds")


df = (
    modelpredictors.filter(pl.col("PredictorName") != "Classifier")
    .group_by("PredictorName")
    .agg(
        Value=pl.when(pl.col("Type").first() == "numeric")
        .then(
            ((pl.col("BinLowerBound") + pl.col("BinUpperBound")) / 2).filter(
                middle_bin()
            )
        )
        .otherwise(
            pl.col("BinSymbol").str.split(",").list.first().filter(middle_bin())
        ),
        Bin=pl.col("BinIndex").filter(middle_bin()),
        Positives=pl.col("BinPositives"),
        Negatives=pl.col("BinNegatives"),
    )
    .with_columns(
        pl.col(["Positives", "Negatives"]).list.get(pl.col("Bin").list.get(0) - 1),
        pl.col("Bin", "Value").list.get(0),
        LogOdds=row_wise_log_odds(
            pl.col("Bin"), pl.col("Positives"), pl.col("Negatives")
        ).round(4),
    )
    .sort("PredictorName")
)

classifier = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")
    .with_columns(
        Propensity=(BinPositives / (BinPositives / BinNegatives)),
        FinalPropensity=((0.5 + BinPositives) / (1 + BinPositives + BinNegatives)),
        ZRatio=cdh_utils.z_ratio(neg_col=BinNegatives, pos_col=BinPositives).round(4),
        Lift=(
            (BinPositives / (BinPositives + BinNegatives))
            / (sumPositives / (BinPositives + BinNegatives).sum())
        ),
    )
    .select(
        [
            pl.col("BinIndex").alias("Index"),
            pl.col("BinSymbol").alias("Bin"),
            BinPositives.alias("Positives"),
            BinNegatives.alias("Negatives"),
            (pl.cum_sum("BinResponseCount") / pl.sum("BinResponseCount")).alias(
                "Cum. Total (%)"
            ),
            (pl.col("BinPropensity")).alias("Propensity (%)"),
            (pl.col("FinalPropensity")).alias("Final Propensity (%)"),
            (pl.cum_sum("BinPositives") / pl.sum("BinPositives")).alias(
                "Cum Positives (%)"
            ),
            pl.col("ZRatio"),
            pl.col("Lift").alias("Lift(%)"),
            pl.col("BinResponseCount").alias("Responses"),
        ]
    )
)
classifierLogOffset = log(1 + classifier["Positives"].sum()) - log(
    1 + classifier["Negatives"].sum()
)

propensity_mapping = df.vstack(
    pl.DataFrame(
        dict(
            zip(
                df.columns,
                ["Final Score"]
                + [None] * 4
                + [(df["LogOdds"].sum() + classifierLogOffset) / (len(df) + 1)],
            )
        ),
        schema=df.schema,
    )
)

GT(propensity_mapping).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="PredictorName")
).tab_style(
    style=style.text(color="blue"), locations=loc.body(columns=["LogOdds"])
).tab_options(table_margin_left=0)

PredictorName,Value,Bin,Positives,Negatives,LogOdds
Customer.Age,34.56,4,9.0,198.0,-1.1459
Customer.AnnualIncome,-24043.049,1,74.0,1166.0,-0.8197
Customer.BusinessSegment,middleSegmentPlus,1,96.0,970.0,-0.3764
Customer.CLV,NON-MISSING,1,111.0,570.0,0.3009
Customer.CLV_VALUE,1345.52,4,31.0,297.0,-0.3227
Customer.CreditScore,518.92,3,33.0,205.0,0.1105
Customer.Date_of_Birth,18773.504,5,28.0,152.0,0.2446
Customer.Gender,U,1,52.0,481.0,-0.2855
Customer.MaritalStatus,No Resp+,1,67.0,745.0,-0.4708
Customer.NetWealth,17992.398,4,51.0,179.0,0.6796


### From Log Odds Score to Final Propensity

The log odds score is mapped to a propensity using a score distribution, which is built using the [PAV(A)](https://en.wikipedia.org/wiki/Isotonic_regression) (Pool Adjacent Violators Algorithm), a form of isotonic regression that ensures the mapping is monotonically increasing.

For each bin (score interval) the final propensity is calculated from the observed positive and negative counts:

$$Final\ Propensity = \frac{0.5 + BinPositives}{1 + BinPositives + BinNegatives}$$

The addition of 0.5 is for Laplace smoothing. An empty model (no responses yet) returns a neutral propensity of 0.5, rather than 0 or undefined. This quickly converges to the observed success rate as more responses are received.

The transformation is a calibrated, isotonic mapping from log odds to propensity, grounded in the actual observed response rates of the model.


In [16]:
# TODO see if we can port the "getActiveRanges" code to python so to highlight the classifier rows that are "active"

GT(classifier.drop("Responses")).tab_style(
    style=style.text(weight="bold"), locations=loc.body(columns="Index")
).tab_style(
    style=style.text(color="blue"),
    locations=loc.body(columns=["Final Propensity (%)"]),
).fmt_percent(pl.selectors.ends_with("(%)")).tab_options(table_margin_left=0)

Index,Bin,Positives,Negatives,Cum. Total (%),Propensity (%),Final Propensity (%),Cum Positives (%),ZRatio,Lift(%)
1,<-0.21,17.0,443.0,28.12%,3.70%,3.80%,8.25%,-9.9945,1.96%
2,"[-0.21, -0.185>",8.0,133.0,36.74%,5.67%,5.99%,12.14%,-3.4954,3.00%
3,"[-0.185, -0.175>",3.0,48.0,39.85%,5.88%,6.73%,13.59%,-1.9775,3.11%
4,"[-0.175, -0.105>",28.0,370.0,64.18%,7.04%,7.14%,27.18%,-4.6281,3.72%
5,"[-0.105, -0.095>",4.0,51.0,67.54%,7.27%,8.04%,29.13%,-1.5054,3.85%
6,"[-0.095, -0.09>",2.0,19.0,68.83%,9.52%,11.36%,30.10%,-0.4788,5.04%
7,"[-0.09, -0.065>",9.0,77.0,74.08%,10.47%,10.92%,34.47%,-0.6578,5.54%
8,"[-0.065, -0.02>",30.0,154.0,85.33%,16.30%,16.49%,49.03%,1.4644,8.63%
9,"[-0.02, 0.03>",37.0,65.0,91.56%,36.27%,36.41%,66.99%,4.913,19.21%
10,"[0.03, 0.06>",20.0,29.0,94.56%,40.82%,41.00%,76.70%,3.664,21.61%


The chart below visualizes the score distribution from the table above. The score bin that contains the calculated final score for the example customer is highlighted.

In [17]:
score = propensity_mapping.filter(PredictorName="Final Score")["LogOdds"][0]
score_bin = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")
    .select(
        pl.col("BinSymbol").filter(
            pl.lit(score).is_between(pl.col("BinLowerBound"), pl.col("BinUpperBound"))
        )
    )
    .item()
)
score_responses = modelpredictors.filter(
    (pl.col("EntryType") == "Classifier") & (pl.col("BinSymbol") == score_bin)
)["BinResponseCount"][0]

score_bin_index = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")["BinSymbol"]
    .to_list()
    .index(score_bin)
)

score_propensity = classifier.row(score_bin_index, named=True)[
    "Final Propensity (%)"
]

final_propensity = (
    modelpredictors.filter(pl.col("EntryType") == "Classifier")
    .with_columns(
        FinalPropensity=(
            (0.5 + BinPositives) / (1 + BinPositives + BinNegatives)
        ).round(5),
    )
    .select(
        pl.col("FinalPropensity").filter(
            (pl.col("BinLowerBound") < score) & (pl.col("BinUpperBound") > score)
        )
    )["FinalPropensity"][0]
)
fig = dm.plot.score_distribution(
    model_id=modelpredictors.get_column("ModelID").unique().to_list()[0]
).add_annotation(
    x=score_bin,
    y=score_propensity / 100,
    text=f"Returned propensity: {score_propensity*100:.2f}%",
    bgcolor="#FFFFFF",
    bordercolor="#000000",
    showarrow=False,
    yref="y2",
    opacity=0.7,
)
bin_index = list(fig.data[0]["x"]).index(score_bin)
fig.data[0]["marker_color"] = (
    ["grey"] * bin_index
    + ["#1f77b4"]
    + ["grey"] * (classifier.shape[0] - bin_index - 1)
)
fig

## Feature Importance

Feature importance in Naive Bayes models represents **how strongly a predictor differentiates between positive and negative outcomes**. It measures the average magnitude of log odds contributions across the predictor's bins.

A predictor with high importance has bins with strong log odds values (far from zero), indicating strong predictive power. A predictor with low importance has bins with weak log odds values (close to zero), indicating weak predictive power.

### Formula

For a predictor with bins $i = 1..n$:

**Step 1: Log odds per bin** (with Laplace smoothing $\frac{1}{n}$):

$$\text{LogOdds}_i = \log\left(Pos_i + \frac{1}{n}\right) - \log\left(\sum Pos + 1\right) - \left[\log\left(Neg_i + \frac{1}{n}\right) - \log\left(\sum Neg + 1\right)\right]$$

**Step 2: Absolute log odds per bin**:

$$|\text{LogOdds}_i|$$

**Step 3: Feature importance** (weighted average of absolute log odds):

$$\text{Importance} = \frac{\sum_i |\text{LogOdds}_i| \times Responses_i}{\sum_i Responses_i}$$

**Step 4: Optional scaling** (default):

$$\text{Scaled Importance} = \frac{\text{Importance}}{\max(\text{Importance})} \times 100$$

### Interpretation

- **Higher values**: Predictor has strong log odds values (strong predictive power)
- **Lower values**: Predictor has weak log odds values (weak predictive power)
- **Zero**: All bins have zero log odds (predictor adds no information)

The absolute log odds captures the magnitude of each bin's contribution to the model score, weighted by how many responses are in that bin. This matches the Pega platform implementation.

In [18]:
# Calculate feature importance for all active predictors
predictor_importance = (
    modelpredictors.filter(pl.col("PredictorName") != "Classifier")
    .with_columns(
        # Calculate both unscaled and scaled importance
        cdh_utils.feature_importance(scaled=False).alias("Importance"),
        cdh_utils.feature_importance(scaled=True).alias("ScaledImportance"),
    )
    .group_by("PredictorName", "ModelID")
    .agg(
        pl.first("Importance"),
        pl.first("ScaledImportance")
    )
    .sort("ScaledImportance", descending=True)
)

GT(predictor_importance.drop("ModelID")).tab_header("Feature Importance by Predictor").fmt_number(
    ["Importance", "ScaledImportance"], decimals=4
)

GT(_tbl_data=shape: (36, 3)
┌─────────────────────────────────┬────────────┬──────────────────┐
│ PredictorName                   ┆ Importance ┆ ScaledImportance │
│ ---                             ┆ ---        ┆ ---              │
│ str                             ┆ f64        ┆ f64              │
╞═════════════════════════════════╪════════════╪══════════════════╡
│ Customer.AnnualIncome           ┆ 0.904282   ┆ 100.0            │
│ Customer.NetWealth              ┆ 0.839899   ┆ 92.880197        │
│ Customer.WinScore               ┆ 0.685932   ┆ 75.853711        │
│ Customer.CreditScore            ┆ 0.649423   ┆ 71.816451        │
│ Customer.RelationshipStartDate  ┆ 0.531624   ┆ 58.789649        │
│ …                               ┆ …          ┆ …                │
│ Customer.pyCountry              ┆ 0.12496    ┆ 13.818747        │
│ IH.Web.Inbound.Rejected.pxLast… ┆ 0.115447   ┆ 12.766695        │
│ Customer.NoOfDependents         ┆ 0.113364   ┆ 12.536352        │
│ IH.SMS.Outbound.Churned.pxLast… ┆ 0.105175   ┆ 11.630731        │
│ IH.SMS.Outbound.Rejected.pxLas… ┆ 0.100734   ┆ 11.139704        │
└─────────────────────────────────┴────────────┴──────────────────┘, _body=<great_tables._gt_data.Body object at 0x7f1d96cc3f50>, _boxhead=Boxhead([ColInfo(var='PredictorName', type=<ColInfoTypeEnum.default: 1>, column_label='PredictorName', column_align='left', column_width=None), ColInfo(var='Importance', type=<ColInfoTypeEnum.default: 1>, column_label='Importance', column_align='right', column_width=None), ColInfo(var='ScaledImportance', type=<ColInfoTypeEnum.default: 1>, column_label='ScaledImportance', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1dee340a90>, _spanners=Spanners([]), _heading=Heading(title='Feature Importance by Predictor', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f1d96ba2050>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f1d96ba2b10>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f1d96ba2c90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7f1d96d36b10>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'

In [19]:
# Visualize feature importance
fig = px.bar(
    predictor_importance,
    x="PredictorName",
    y="ScaledImportance",
    title="Predictor Feature Importance (Scaled 0-100)",
    labels={"ScaledImportance": "Importance", "PredictorName": ""},
    template="pega",
)
fig.update_layout(
    xaxis_tickangle=-45,
    height=500,
    margin=dict(b=180)  # Increase bottom margin for longer rotated labels
)
fig.update_xaxes(tickfont=dict(size=10))  # Smaller font for x-axis labels
fig.show()